In [2]:
import torch
import numpy as np
import pandas as pd
import av
from pathlib import Path 
import matplotlib.pyplot as plt
from torch.utils.data import Dataset

In [3]:
vod_title = "M8 vs. EDG - VALORANT Masters Santiago - SWISS"
video_path = Path(f"../../data/vods/{vod_title}.mp4")
csv_path = Path(f"../../data/processed/round_labels/{vod_title}_round_labels_fixed.csv")
round_df = pd.read_csv(csv_path)

In [4]:
import yaml
CONFIG_PATH = "../../configs/laptop.yaml" 
#環境によってパスを変える!!

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

device = cfg["runtime"]["device"]
sample_mode = cfg["runtime"]["sample_mode"]
use_gpu = cfg["runtime"]["use_gpu"]
print(cfg)


{'runtime': {'device': 'cpu', 'sample_mode': True, 'use_gpu': False}}


In [8]:
small_df = round_df.head(3).copy()
video_id = "72YPJbZQMdU"
small_df["video_id"] = video_id
cache_dir = Path("../../data/cache/test")
cache_dir.mkdir(parents=True,exist_ok=True)

In [10]:
small_df

,value,start_sec,end_sec,n_samples,mean_diff,min_diff,max_diff,duration_sec,t_sec,t_min,...,fix_applied,needs_review,map,start_timer_sec,start_search_offset_sec,search_start_sec,start_timer_raws,attacker_win,defender_win,video_id
0,True,9.009,78.078,70,22.858459,17.960205,26.488715,69.069,9.009,0.150150,...,NaN,False,HAVEN,15.0,14.0,23.009,"[16.0, 15.0, 15.0, 15.0, 15.0]",0,1,72YPJbZQMdU
1,True,106.106,203.203,98,23.621082,19.780490,27.049723,97.097,106.106,1.768433,...,NaN,False,HAVEN,6.0,5.0,111.106,"[6.0, 6.0, 6.0, 6.0, 6.0]",1,0,72YPJbZQMdU
2,True,235.235,278.278,44,25.698283,21.291341,27.700168,43.043,235.235,3.920583,...,NaN,False,HAVEN,3.0,2.0,237.235,"[3.0, 3.0, 3.0]",1,0,72YPJbZQMdU


In [ ]:
import torch

for _,row in small_df.iterrows():
    #TODO 便利関数用意する

In [6]:
class RoundCacheDataset(Dataset):
    def __init__(self,df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self,idx):
        row = self.df.iloc[idx]
        cache_path = row["cache_path"]
        pixel_values = torch.load(cache_path,map_location="cpu")
        label = int(row["attacker_win"])

        return {
            "pixel_values":pixel_values,
            "labels":torch.tensor(label).long(),
        }
        